# KNN Purchase Probability

전처리 노트북에서 저장한 KNN용 split 데이터로 구매 가능성 점수를 만든다.

| 실험 | 역할 |
|---|---|
| raw KNN baseline | `log1p`, `StandardScaler`, `SMOTE` 없이 원본 단위 거리만 사용 |
| Pipeline KNN | `log1p -> StandardScaler -> SMOTE -> KNN`을 CV fold 내부에서 적용 |
| 비교 기준 | raw baseline 대비 Pipeline의 순위화 성능, 탐지 폭, 확률 보정 변화 |

## 0. KNN Pipeline 입력 데이터 로드

In [30]:
import pandas as pd

# 전처리 노트북에서 저장한 KNN용 split 데이터를 불러온다.
# 이 CSV는 SMOTE/Scaler 적용 전 데이터라서, 모델 전처리는 아래 Pipeline에서 따로 수행한다.
X_train_knn = pd.read_csv("csv/X_train_knn.csv")
X_test_knn = pd.read_csv("csv/X_test_knn.csv")
y_train_knn = pd.read_csv("csv/y_train_knn.csv")["Revenue"]
y_test_knn = pd.read_csv("csv/y_test_knn.csv")["Revenue"]

# 데이터 크기와 양성 비율을 확인해, 중복 제거 후 split 데이터가 정상 로드됐는지 점검한다.
print(f"X_train_knn: {X_train_knn.shape}")
print(f"X_test_knn : {X_test_knn.shape}")
print(f"y_train_knn: {y_train_knn.shape}, positive_ratio = {y_train_knn.mean():.4f}")
print(f"y_test_knn : {y_test_knn.shape}, positive_ratio = {y_test_knn.mean():.4f}")
print("\nColumns")
print(list(X_train_knn.columns))


X_train_knn: (9764, 11)
X_test_knn : (2441, 11)
y_train_knn: (9764,), positive_ratio = 0.1563
y_test_knn : (2441,), positive_ratio = 0.1565

Columns
['Administrative', 'Administrative_Duration', 'Informational', 'Informational_Duration', 'ProductRelated', 'ProductRelated_Duration', 'BounceRates', 'ExitRates', 'SpecialDay', 'Weekend', 'is_new_visitor']


**읽는 법**

- `X_train_knn`이 9,764행이고 양성 비율이 약 15.6%면 SMOTE 전 split 데이터가 정상 로드된 것이다.
- 이 불균형 상태가 raw baseline과 Pipeline의 공통 출발점이다.

## 1. 기준 KNN 모델

`log1p`, `StandardScaler`, `SMOTE`를 쓰지 않은 raw KNN을 기준선으로 둔다. 중복 제거와 `PageValues` 제거는 누수 방지용 데이터 정의이므로 유지한다.

In [31]:
import numpy as np

from imblearn.over_sampling import SMOTE
from imblearn.pipeline import Pipeline
from sklearn.metrics import average_precision_score, brier_score_loss, f1_score, precision_score, recall_score, roc_auc_score
from sklearn.neighbors import KNeighborsRegressor
from sklearn.preprocessing import FunctionTransformer, StandardScaler

RANDOM_STATE = 42

# KNN 거리 계산에서 큰 값이 지배하지 않도록 log1p를 적용할 우편향 수치형 컬럼을 지정한다.
KNN_LOG_COLS = [
    "Administrative", "Administrative_Duration",
    "Informational", "Informational_Duration",
    "ProductRelated", "ProductRelated_Duration",
    "BounceRates", "ExitRates", "SpecialDay",
]

# Pipeline 안에서 호출할 log1p 변환 함수다. fold마다 train 데이터에만 fit/transform 흐름이 적용되도록 함수로 묶는다.
def log1p_selected_columns(X):
    X = X.copy()
    X[KNN_LOG_COLS] = np.log1p(X[KNN_LOG_COLS])
    return X

# KNN용 모델 전처리를 한 번에 묶는다. GridSearchCV가 이 Pipeline 전체를 fold별로 새로 fit하므로 SMOTE 누수를 막을 수 있다.
def build_knn_pipeline(n_neighbors=5, weights="uniform", p=2):
    return Pipeline([
        ("log1p", FunctionTransformer(log1p_selected_columns, validate=False)),
        ("scaler", StandardScaler()),
        ("smote", SMOTE(random_state=RANDOM_STATE, k_neighbors=5)),
        ("knn", KNeighborsRegressor(
            n_neighbors=n_neighbors,
            weights=weights,
            p=p,
        )),
    ])

BASE_K = 5

# 아무 모델 전처리도 하지 않은 raw baseline이다. 이후 Pipeline 모델과 비교해 전처리 효과를 확인한다.
raw_baseline_model = KNeighborsRegressor(n_neighbors=BASE_K, weights="uniform", p=2)
raw_baseline_model.fit(X_train_knn, y_train_knn)
raw_baseline_proba = raw_baseline_model.predict(X_test_knn)
raw_baseline_pred = (raw_baseline_proba >= 0.5).astype(int)

# raw baseline의 테스트 성능을 표로 정리한다. 이 값이 전처리 적용 전 기준선이다.
raw_baseline_metrics = pd.DataFrame([
    {
        "model": "raw baseline",
        "ROC_AUC": roc_auc_score(y_test_knn, raw_baseline_proba),
        "PR_AUC": average_precision_score(y_test_knn, raw_baseline_proba),
        "Brier": brier_score_loss(y_test_knn, raw_baseline_proba),
        "Precision": precision_score(y_test_knn, raw_baseline_pred, zero_division=0),
        "Recall": recall_score(y_test_knn, raw_baseline_pred),
        "F1": f1_score(y_test_knn, raw_baseline_pred),
    }
])

# raw baseline의 예측값이 어떤 범위와 분포를 갖는지 먼저 확인한다.
print("Raw baseline sample predictions")
print(raw_baseline_proba[:10])

print("\nRaw baseline prediction summary")
display(pd.Series(raw_baseline_proba, name="raw_baseline_proba").describe().to_frame().T)

print("Raw baseline test metrics")
display(raw_baseline_metrics.round(4))


Raw baseline sample predictions
[0.4 0.  0.2 0.  0.  0.6 0.  0.4 0.  0. ]

Raw baseline prediction summary


,count,mean,std,min,25%,50%,75%,max
raw_baseline_proba,2441.0,0.154609,0.18054,0.0,0.0,0.2,0.2,1.0


Raw baseline test metrics


,model,ROC_AUC,PR_AUC,Brier,Precision,Recall,F1
0,raw baseline,0.6173,0.2084,0.1438,0.3016,0.0995,0.1496


**읽는 법**

- raw baseline은 원본 단위 거리만 사용한 기준선이다.
- ROC-AUC, PR-AUC, Recall이 낮으면 모델 전처리 없이 구매자를 잘 끌어올리지 못한다는 뜻이다.

## 2. K 후보 탐색

K는 몇 명의 이웃을 평균낼지 정하는 값이다. 작은 K는 세부 패턴에 민감하고, 큰 K는 안정적이지만 개별 패턴이 흐려질 수 있다.

```python
K_CANDIDATES = [1, 3, 5, 7, 11, 15, 21, 31, 51, 75, 101, 151]
```

| 구간 | 후보 | 보는 이유 |
|---|---|---|
| 작은 K | `1`, `3`, `5`, `7` | 가까운 몇 명만 보는 구간이다. 세부 패턴은 잘 잡지만 노이즈에도 민감하다. |
| 중간 K | `11`, `15`, `21`, `31` | 너무 예민한 상태에서 조금씩 안정화되는 구간이다. |
| 큰 K | `51`, `75`, `101`, `151` | 많은 이웃을 평균내는 구간이다. 안정적이지만 개별 패턴은 흐려질 수 있다. |

큰 K가 선택되면 KNN이 소수의 가까운 이웃보다 넓은 평균에 기대고 있다는 신호로 해석한다.

## 2-1. 직접 반복문으로 K 비교

`StratifiedKFold`로 구매 비율을 유지하고, 각 fold 안에서 Pipeline을 새로 fit한다. 핵심은 SMOTE가 검증 fold를 보지 못하게 하는 것이다.

| 지표 | 의미 | 해석 방향 |
|---|---|---|
| ROC-AUC | 구매자를 비구매자보다 높은 점수로 올리는지 | 높을수록 좋음 |
| PR-AUC | 구매자가 적은 상황에서 상위 점수에 구매자가 잘 섞이는지 | 높을수록 좋음 |
| Brier | 예측 확률과 실제값의 차이 | 낮을수록 좋음 |
| Precision | 구매 예측 중 실제 구매 비율 | 높을수록 좋음 |
| Recall | 실제 구매자 중 잡아낸 비율 | 높을수록 좋음 |
| F1 | Precision과 Recall의 균형 | 높을수록 좋음 |

In [32]:
from sklearn.model_selection import StratifiedKFold

# 작은 K부터 큰 K까지 후보를 두어, KNN이 국소 이웃에 기대는지 넓은 평균에 기대는지 확인한다.
K_CANDIDATES = [1, 3, 5, 7, 11, 15, 21, 31, 51, 75, 101, 151]

# 구매/미구매 비율이 fold마다 비슷하게 유지되도록 stratified 5-fold를 사용한다.
cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=RANDOM_STATE)
rows = []

# 먼저 uniform + Euclidean 조건에서 K 변화만 직접 비교해, K가 커질수록 성능이 어떻게 움직이는지 본다.
for k in K_CANDIDATES:
    auc_scores = []
    ap_scores = []
    brier_scores = []

    for train_idx, valid_idx in cv.split(X_train_knn, y_train_knn):
        X_tr = X_train_knn.iloc[train_idx]
        X_va = X_train_knn.iloc[valid_idx]
        y_tr = y_train_knn.iloc[train_idx]
        y_va = y_train_knn.iloc[valid_idx]

        # 각 fold 안에서 log1p, scaling, SMOTE, KNN을 함께 fit한다.
        model = build_knn_pipeline(n_neighbors=k, weights="uniform", p=2)
        model.fit(X_tr, y_tr)
        proba = model.predict(X_va)

        auc_scores.append(roc_auc_score(y_va, proba))
        ap_scores.append(average_precision_score(y_va, proba))
        brier_scores.append(brier_score_loss(y_va, proba))

    rows.append({
        "K": k,
        "ROC_AUC": np.mean(auc_scores),
        "PR_AUC": np.mean(ap_scores),
        "Brier": np.mean(brier_scores),
    })

# K 후보별 평균 성능을 하나의 표로 정리한다.
k_result = pd.DataFrame(rows).sort_values("ROC_AUC", ascending=False).reset_index(drop=True)

print("Pipeline K comparison: log1p + scaler + SMOTE + uniform Euclidean KNN")
display(k_result.round(4))


Pipeline K comparison: log1p + scaler + SMOTE + uniform Euclidean KNN


,K,ROC_AUC,PR_AUC,Brier
0,101,0.7226,0.2817,0.2388
1,75,0.7224,0.2779,0.2390
2,151,0.7216,0.2837,0.2386
3,51,0.7213,0.2772,0.2389
4,31,0.7189,0.2795,0.2380
5,21,0.7141,0.2764,0.2376
6,15,0.7077,0.2693,0.2372
7,11,0.6984,0.2595,0.2390
8,7,0.6828,0.2483,0.2423
9,5,0.6690,0.2350,0.2466


**읽는 법**

- 이 표는 `uniform + Euclidean` 조건에서 K만 바꾼 결과다.
- 큰 K가 위쪽에 몰리면, 모델이 가까운 소수 이웃보다 많은 이웃의 평균으로 안정화된다는 뜻이다.

**지표 기준:** ROC-AUC와 PR-AUC를 중심으로 읽고, Brier와 분류 지표는 보조적으로 확인한다.

### 2-2. GridSearchCV로 K 찾기

`GridSearchCV`에는 Pipeline 전체를 넘겨 K, 거리 기준, 가중치 방식을 함께 탐색한다. 따라서 `log1p`, `StandardScaler`, `SMOTE`도 각 train fold 안에서만 적용된다.

In [33]:
from sklearn.metrics import make_scorer
from sklearn.model_selection import GridSearchCV

# GridSearchCV의 선택 기준은 구매자 순위를 잘 올리는 ROC-AUC로 둔다.
auc_scorer = make_scorer(roc_auc_score, response_method="predict")

# K뿐 아니라 거리 기준(p)과 이웃 가중치 방식까지 함께 탐색한다.
param_grid = {
    "knn__n_neighbors": K_CANDIDATES,
    "knn__weights": ["uniform", "distance"],
    "knn__p": [1, 2],
}

# estimator로 Pipeline 전체를 넘겨야 log1p, scaling, SMOTE가 fold 내부에서만 수행된다.
grid = GridSearchCV(
    estimator=build_knn_pipeline(),
    param_grid=param_grid,
    scoring=auc_scorer,
    cv=cv,
    n_jobs=-1,
)

grid.fit(X_train_knn, y_train_knn)

# cv_results_ 전체를 DataFrame으로 바꾼 뒤, 보고서에 필요한 핵심 컬럼만 남긴다.
grid_result_df = pd.DataFrame(grid.cv_results_)
grid_result_df = grid_result_df[[
    "param_knn__n_neighbors",
    "param_knn__weights",
    "param_knn__p",
    "mean_test_score",
    "std_test_score",
    "rank_test_score",
]].rename(columns={
    "param_knn__n_neighbors": "K",
    "param_knn__weights": "weights",
    "param_knn__p": "p",
    "mean_test_score": "CV_ROC_AUC",
    "std_test_score": "CV_ROC_AUC_std",
    "rank_test_score": "rank",
})

# p값만 보면 직관적이지 않으므로, 거리 기준 이름을 붙여서 출력 표를 읽기 쉽게 만든다.
grid_result_df["distance"] = grid_result_df["p"].map({1: "Manhattan", 2: "Euclidean"})
grid_result_df["setting"] = grid_result_df["distance"] + " + " + grid_result_df["weights"]
grid_result_df = grid_result_df[
    ["setting", "weights", "p", "K", "CV_ROC_AUC", "CV_ROC_AUC_std", "rank"]
].sort_values("CV_ROC_AUC", ascending=False).reset_index(drop=True)

# 설정별 최고 결과만 따로 뽑아 전체 출력이 너무 길어지지 않게 한다.
grid_best_df = (
    grid_result_df
    .sort_values(["setting", "CV_ROC_AUC"], ascending=[True, False])
    .groupby("setting", as_index=False)
    .head(1)
    .sort_values("CV_ROC_AUC", ascending=False)
    .reset_index(drop=True)
)

print("Best GridSearchCV result by setting")
display(grid_best_df.round(4))

print("Top 8 GridSearchCV candidates")
display(grid_result_df.head(8).round(4))


Best GridSearchCV result by setting


,setting,weights,p,K,CV_ROC_AUC,CV_ROC_AUC_std,rank
0,Manhattan + distance,distance,1,151,0.7316,0.0166,1
1,Euclidean + distance,distance,2,151,0.7277,0.0163,4
2,Manhattan + uniform,uniform,1,101,0.7272,0.0134,5
3,Euclidean + uniform,uniform,2,101,0.7226,0.0131,13


Top 8 GridSearchCV candidates


,setting,weights,p,K,CV_ROC_AUC,CV_ROC_AUC_std,rank
0,Manhattan + distance,distance,1,151,0.7316,0.0166,1
1,Manhattan + distance,distance,1,101,0.7312,0.0128,2
2,Manhattan + distance,distance,1,75,0.7287,0.0117,3
3,Euclidean + distance,distance,2,151,0.7277,0.0163,4
4,Manhattan + uniform,uniform,1,101,0.7272,0.0134,5
5,Euclidean + distance,distance,2,101,0.7271,0.0132,6
6,Manhattan + uniform,uniform,1,151,0.7263,0.0178,7
7,Euclidean + distance,distance,2,75,0.7258,0.0119,8


**읽는 법**

- `grid_best_df`: 거리 기준과 가중치 방식별 최고 조합만 압축해서 본다.
- `Top 8`: 큰 K 후보가 반복해서 상위권이면, 가까운 소수 이웃보다 넓은 이웃 평균이 더 안정적이라는 신호다.

### 2-3. 결과 분석 및 K값 결정

최종 모델은 `grid_result_df`에서 CV ROC-AUC가 가장 높은 1위 조합을 사용한다.

## 3. 최종 모델 재학습 및 테스트 평가

교차검증에서 가장 좋았던 Pipeline 조합을 전체 학습 데이터에 다시 fit하고, 테스트셋에서 최종 성능을 확인한다.

In [34]:
from sklearn.metrics import f1_score, precision_score, recall_score

# GridSearchCV 결과표에서 가장 높은 CV ROC-AUC를 보인 조합을 최종 설정으로 선택한다.
best_row = grid_result_df.iloc[0]
best_k = int(best_row["K"])
best_weights = best_row["weights"]
best_p = int(best_row["p"])
best_distance = best_row["setting"].split(" + ")[0]

# 선택된 설정으로 최종 Pipeline을 만들고 전체 학습 데이터에 다시 fit한다.
final_model = build_knn_pipeline(
    n_neighbors=best_k,
    weights=best_weights,
    p=best_p,
)
final_model.fit(X_train_knn, y_train_knn)

# 테스트셋은 모델 선택에 쓰지 않았으므로, 여기서 최종 일반화 성능을 확인한다.
test_proba = final_model.predict(X_test_knn)
test_pred = (test_proba >= 0.5).astype(int)

# 확률 순위 품질과 0.5 임계값 기준 분류 지표를 함께 정리한다.
final_metrics = pd.DataFrame([
    {
        "model": "pipeline selected",
        "ROC_AUC": roc_auc_score(y_test_knn, test_proba),
        "PR_AUC": average_precision_score(y_test_knn, test_proba),
        "Brier": brier_score_loss(y_test_knn, test_proba),
        "Precision": precision_score(y_test_knn, test_pred, zero_division=0),
        "Recall": recall_score(y_test_knn, test_pred),
        "F1": f1_score(y_test_knn, test_pred),
    }
])

print("Final Pipeline model")
print(f"K = {best_k}, weights = {best_weights}, p = {best_p} ({best_distance})")
print(f"test predicted probability range: {test_proba.min():.4f} ~ {test_proba.max():.4f}")
print(f"test mean predicted probability: {test_proba.mean():.4f}")
print(f"test actual purchase ratio: {y_test_knn.mean():.4f}")

display(final_metrics.round(4))


Final Pipeline model
K = 151, weights = distance, p = 1 (Manhattan)
test predicted probability range: 0.0000 ~ 1.0000
test mean predicted probability: 0.4578
test actual purchase ratio: 0.1565


,model,ROC_AUC,PR_AUC,Brier,Precision,Recall,F1
0,pipeline selected,0.7284,0.2988,0.2352,0.2427,0.8246,0.375


**읽는 법**

- ROC-AUC, PR-AUC: 구매 가능성이 높은 세션을 위쪽으로 올리는 능력이다.
- Recall이 높고 Precision이 낮으면 구매 후보를 넓게 잡지만 오탐도 많다는 뜻이다.
- Brier는 낮을수록 좋다. 높게 나오면 예측값을 실제 구매 확률로 읽기 어렵다.

## 4. 기준 모델 대비 개선 분석

두 표만 본다. `cv_compare`는 Pipeline 내부 튜닝 효과를 보여주고, `test_compare`는 무전처리 raw baseline과 최종 Pipeline의 실제 테스트 성능 차이를 보여준다.

In [35]:
# Pipeline 안에서 기준 설정(K=5, uniform, Euclidean)과 최종 선택 설정을 비교한다.
baseline_like_row = grid_result_df[
    (grid_result_df["K"] == BASE_K)
    & (grid_result_df["weights"] == "uniform")
    & (grid_result_df["p"] == 2)
].iloc[0]

cv_compare = pd.DataFrame([
    {
        "model": "pipeline baseline-like",
        "K": int(baseline_like_row["K"]),
        "weights": baseline_like_row["weights"],
        "p": int(baseline_like_row["p"]),
        "setting": baseline_like_row["setting"],
        "CV_ROC_AUC": baseline_like_row["CV_ROC_AUC"],
    },
    {
        "model": "pipeline selected",
        "K": best_k,
        "weights": best_weights,
        "p": best_p,
        "setting": best_row["setting"],
        "CV_ROC_AUC": best_row["CV_ROC_AUC"],
    },
])

# 테스트셋에서는 무전처리 raw baseline과 최종 Pipeline을 나란히 비교해 전처리 효과를 확인한다.
test_compare = pd.concat([
    raw_baseline_metrics.assign(preprocessing="none"),
    final_metrics.assign(preprocessing="log1p + scaler + SMOTE"),
], ignore_index=True)

# raw baseline 대비 최종 Pipeline의 테스트 지표 변화량과 퍼센트 변화를 계산한다.
metric_cols = ["ROC_AUC", "PR_AUC", "Brier", "Precision", "Recall", "F1"]
raw_metrics = test_compare.loc[test_compare["model"] == "raw baseline", metric_cols].iloc[0]
pipeline_metrics = test_compare.loc[test_compare["model"] == "pipeline selected", metric_cols].iloc[0]

test_metric_change = pd.DataFrame({
    "metric": metric_cols,
    "raw_baseline": raw_metrics.values,
    "pipeline": pipeline_metrics.values,
})
test_metric_change["absolute_change"] = (
    test_metric_change["pipeline"] - test_metric_change["raw_baseline"]
)
test_metric_change["percent_change"] = (
    test_metric_change["absolute_change"] / test_metric_change["raw_baseline"] * 100
)

# Brier는 낮을수록 좋고, 나머지 지표는 높을수록 좋다.
test_metric_change["direction"] = test_metric_change["metric"].map({
    "Brier": "lower is better",
}).fillna("higher is better")

cv_roc_change = pd.DataFrame([{
    "metric": "CV_ROC_AUC",
    "baseline_like": cv_compare.loc[0, "CV_ROC_AUC"],
    "selected": cv_compare.loc[1, "CV_ROC_AUC"],
}])
cv_roc_change["absolute_change"] = cv_roc_change["selected"] - cv_roc_change["baseline_like"]
cv_roc_change["percent_change"] = cv_roc_change["absolute_change"] / cv_roc_change["baseline_like"] * 100

print("CV comparison inside Pipeline search")
display(cv_compare.round(4))

print("CV ROC-AUC change")
display(cv_roc_change.round(4))

print("Test comparison: raw baseline vs selected Pipeline")
display(test_compare.round(4))

print("Test metric change: raw baseline -> selected Pipeline")
display(test_metric_change.round(4))


CV comparison inside Pipeline search


,model,K,weights,p,setting,CV_ROC_AUC
0,pipeline baseline-like,5,uniform,2,Euclidean + uniform,0.6690
1,pipeline selected,151,distance,1,Manhattan + distance,0.7316


Test comparison: raw baseline vs selected Pipeline


,model,ROC_AUC,PR_AUC,Brier,Precision,Recall,F1,preprocessing
0,raw baseline,0.6173,0.2084,0.1438,0.3016,0.0995,0.1496,none
1,pipeline selected,0.7284,0.2988,0.2352,0.2427,0.8246,0.3750,log1p + scaler + SMOTE


**읽는 법**

- CV ROC-AUC: `0.6690 -> 0.7316`, 약 `+0.0626`p, 비율로는 `+9.4%` 개선이다.
- 테스트 ROC-AUC: raw baseline 대비 약 `+18.0%` 개선되어 순위화 성능이 좋아졌다.
- 테스트 PR-AUC: 약 `+43.3%` 개선되어 불균형 데이터에서 구매자 상위 포착력이 좋아졌다.
- Recall과 F1은 크게 올랐지만 Precision은 낮아졌다. 구매자를 더 많이 잡는 대신 오탐이 늘어난 구조다.
- Brier는 약 `+63.5%` 나빠졌다. 예측값을 보정된 실제 구매 확률로 해석하면 안 된다.

## 5. KNN 결과로 확인할 수 있는 인사이트

여기서는 성능 지표보다 모델 점수의 의미를 해석한다. 예측 점수가 실제 구매율을 얼마나 잘 정렬하는지, 어떤 행동 패턴을 구매 신호로 보는지, 그리고 KNN 확률값을 어디까지 믿을 수 있는지 확인한다.

### 5-1. 예측 확률 구간별 실제 구매율

예측 확률을 10개 구간으로 나누어, 상위 구간일수록 실제 구매율이 높아지는지 확인한다.

In [36]:
# 테스트셋의 예측 확률과 실제 구매 여부를 하나의 표로 정리
test_result = pd.DataFrame({
    "predicted_proba": test_proba,
    "actual_revenue": y_test_knn,
})

# score_decile: 예측 확률 순위 기준 10개 구간, 10이 가장 높은 구간
test_result["score_decile"] = pd.qcut(
    test_result["predicted_proba"].rank(method = "first"),
    q = 10,
    labels = range(1, 11),
).astype(int)

# 예측 점수 구간별 실제 구매율과 구매자 수 집계
decile_result = (
    test_result
    .groupby("score_decile")
    .agg(
        count = ("actual_revenue", "size"),
        mean_predicted_proba = ("predicted_proba", "mean"),
        actual_purchase_rate = ("actual_revenue", "mean"),
        actual_buyers = ("actual_revenue", "sum"),
    )
    .reset_index()
)

# 상위 구간으로 갈수록 실제 구매율이 높아지는지 확인
display(decile_result)

,score_decile,count,mean_predicted_proba,actual_purchase_rate,actual_buyers
0,1,245,0.009343,0.012245,3
1,2,244,0.091665,0.020492,5
2,3,244,0.247475,0.045082,11
3,4,244,0.398169,0.081967,20
4,5,244,0.487958,0.180328,44
5,6,244,0.557409,0.143443,35
6,7,244,0.616104,0.237705,58
7,8,244,0.666054,0.262295,64
8,9,244,0.715996,0.270492,66
9,10,244,0.789169,0.311475,76


**읽는 법**

- 이 표는 예측 확률을 10개 구간으로 나눈 뒤, 각 구간의 실제 구매율을 확인한 것이다. 즉, 모델 점수가 높은 그룹이 실제로도 더 많이 구매했는지 보는 순위 검증이다.
- 현재 결과는 1구간 실제 구매율이 약 `1.2%`, 10구간이 약 `31.1%`다. 최상위 구간의 구매율이 최하위 구간보다 약 25배 높으므로, 모델 점수는 구매 가능성이 높은 세션을 위쪽으로 올리는 데 의미가 있다.
- 다만 모든 구간이 완벽하게 단조 증가하지는 않는다. 예를 들어 5구간보다 6구간 구매율이 낮다. 따라서 점수의 큰 방향성은 믿을 수 있지만, `0.56이면 실제 구매확률 56%`처럼 개별 확률값을 그대로 해석하면 안 된다.

### 5-2. 예측 상위 그룹과 하위 그룹의 행동 차이

예측 확률 상위 10%와 하위 10%의 원래 단위 행동 지표 평균을 비교한다.

In [37]:
# 상위 10%, 하위 10%를 추출하기 위한 boolean mask 생성
top_group = test_result["score_decile"] == 10
bottom_group = test_result["score_decile"] == 1

# 상위/하위 그룹의 원래 단위 입력 변수 평균 계산
top_profile = X_test_knn.loc[top_group].mean()
bottom_profile = X_test_knn.loc[bottom_group].mean()

# 두 그룹의 평균 차이를 하나의 표로 정리
profile_compare = pd.DataFrame({
    "top_10pct_mean": top_profile,
    "bottom_10pct_mean": bottom_profile,
})
profile_compare["difference"] = profile_compare["top_10pct_mean"] - profile_compare["bottom_10pct_mean"]
profile_compare["abs_difference"] = profile_compare["difference"].abs()

# 차이가 큰 변수부터 확인할 수 있도록 정렬
profile_compare = profile_compare.sort_values("abs_difference", ascending = False)

# 예측 상위 그룹과 하위 그룹의 행동 차이 출력
display(profile_compare)

,top_10pct_mean,bottom_10pct_mean,difference,abs_difference
ProductRelated_Duration,2827.780186,66.734091,2761.046095,2761.046095
Administrative_Duration,162.654130,0.081633,162.572498,162.572498
Informational_Duration,84.122119,0.093878,84.028242,84.028242
ProductRelated,71.758197,3.975510,67.782687,67.782687
Administrative,4.135246,0.008163,4.127083,4.127083
Informational,1.139344,0.020408,1.118936,1.118936
is_new_visitor,0.356557,0.016327,0.340231,0.340231
SpecialDay,0.011475,0.164082,-0.152606,0.152606
ExitRates,0.014575,0.146582,-0.132007,0.132007
Weekend,0.258197,0.142857,0.115340,0.115340


**읽는 법**

- 이 표는 예측 상위 10%와 하위 10%가 실제 행동 지표에서 얼마나 다른지 보여준다. KNN이 높은 점수를 준 세션이 어떤 모습인지 설명하는 용도다.
- 상위 그룹은 `ProductRelated_Duration`이 약 `2827.8`, 하위 그룹은 약 `66.7`이다. 상품 관련 페이지에 머문 시간이 압도적으로 길기 때문에, 모델은 상품 탐색 강도를 강한 구매 신호로 본다.
- `ProductRelated`도 상위 그룹이 약 `71.8`, 하위 그룹이 약 `4.0`으로 차이가 크다. 단순 방문보다 여러 상품 페이지를 오래 탐색한 세션이 높은 점수를 받는다.
- 반대로 `ExitRates`, `BounceRates`는 상위 그룹에서 낮다. 높은 점수를 받은 사용자는 바로 이탈하기보다 사이트 안에서 더 오래 움직인 사용자에 가깝다.
- `is_new_visitor` 차이도 크다. 이 결과만 보면 신규 방문자 여부가 모델 점수에 영향을 주지만, 이것이 곧 신규 방문자가 무조건 구매 확률이 높다는 인과관계를 뜻하지는 않는다.

### 5-3. Permutation Importance

컬럼을 하나씩 섞었을 때 ROC-AUC가 얼마나 떨어지는지 확인해, KNN이 의존한 예측 신호를 본다.

In [38]:
from sklearn.inspection import permutation_importance

# permutation importance도 ROC-AUC 감소량 기준으로 계산
permutation_scorer = make_scorer(roc_auc_score, response_method = "predict")

# 각 컬럼을 반복적으로 섞어 모델 성능 감소량 측정
importance = permutation_importance(
    final_model,
    X_test_knn,
    y_test_knn,
    scoring = permutation_scorer,
    n_repeats = 5,
    random_state = 42,
)

# 컬럼별 중요도 평균과 표준편차를 표로 정리
importance_result = pd.DataFrame({
    "feature": X_test_knn.columns,
    "importance_mean": importance.importances_mean,
    "importance_std": importance.importances_std,
})

# 성능 감소가 큰 변수부터 확인할 수 있도록 정렬
importance_result = importance_result.sort_values("importance_mean", ascending = False)

# KNN 모델이 많이 의존한 예측 신호 출력
display(importance_result)

,feature,importance_mean,importance_std
10,is_new_visitor,0.031544,0.006249
5,ProductRelated_Duration,0.021094,0.008069
7,ExitRates,0.019201,0.004082
4,ProductRelated,0.013856,0.003195
6,BounceRates,0.004052,0.004344
1,Administrative_Duration,0.003312,0.003525
8,SpecialDay,0.002750,0.004105
0,Administrative,0.000434,0.003368
3,Informational_Duration,-0.008246,0.002192
9,Weekend,-0.008749,0.003096


**읽는 법**

- Permutation Importance는 특정 컬럼을 섞어서 그 정보만 망가뜨렸을 때 ROC-AUC가 얼마나 떨어지는지 보는 방식이다. 많이 떨어질수록 모델이 그 컬럼에 더 의존했다는 뜻이다.
- 현재는 `is_new_visitor`, `ProductRelated_Duration`, `ExitRates`, `ProductRelated`가 위에 있다. 즉, 최종 KNN은 방문자 유형, 상품 탐색 시간, 이탈률, 상품 페이지 수를 핵심 거리 신호로 사용한다.
- 이 결과는 5-2의 상·하위 그룹 비교와도 연결된다. 높은 점수를 받은 그룹이 상품 페이지를 많이 보고 오래 머물렀고, permutation에서도 같은 계열 변수가 중요하게 나온다.
- `Informational`, `Weekend`처럼 음수로 나온 변수는 이 테스트셋에서는 섞어도 성능이 떨어지지 않았거나 오히려 약간 나아졌다는 뜻이다. 이런 변수는 KNN 거리 계산에서 도움이 불안정하거나 잡음처럼 작동했을 가능성이 있다.

### 5-4. KNN 이웃 예시 확인

예측 확률이 가장 높은 방문자 1명을 골라, 최종 KNN 공간에서 가까운 이웃 일부를 확인한다.

In [39]:
# 예측 확률이 가장 높은 테스트 방문자 1명을 선택한다.
target_idx = int(test_result.sort_values("predicted_proba", ascending=False).index[0])

# Pipeline 앞단 변환을 직접 통과시켜, 최종 KNN 단계가 보는 좌표계로 변환한다.
target_after_log = final_model.named_steps["log1p"].transform(X_test_knn.iloc[[target_idx]])
target_scaled = final_model.named_steps["scaler"].transform(target_after_log)

# SMOTE 이후 학습 공간에서 가장 가까운 이웃을 찾는다.
knn_step = final_model.named_steps["knn"]
distances, neighbor_idx = knn_step.kneighbors(target_scaled, n_neighbors=best_k)

neighbor_result = pd.DataFrame({
    "neighbor_rank": range(1, best_k + 1),
    "distance": distances[0],
    "neighbor_revenue": knn_step._y[neighbor_idx[0]],
})

# 이웃 표 전체를 다 보기보다, 예측을 해석하는 핵심 요약 지표를 먼저 만든다.
neighbor_summary = pd.DataFrame([{
    "target_test_index": target_idx,
    "predicted_proba": test_result.loc[target_idx, "predicted_proba"],
    "actual_revenue": test_result.loc[target_idx, "actual_revenue"],
    "K": best_k,
    "buyers_among_K": neighbor_result["neighbor_revenue"].sum(),
    "buyer_ratio_among_K": neighbor_result["neighbor_revenue"].mean(),
    "nearest_distance": neighbor_result["distance"].min(),
    "median_distance": neighbor_result["distance"].median(),
    "max_distance": neighbor_result["distance"].max(),
}])

print(f"target test index: {target_idx}")
print(f"predicted purchase probability: {test_result.loc[target_idx, 'predicted_proba']:.4f}")
print(f"actual revenue: {test_result.loc[target_idx, 'actual_revenue']}")
print(f"buyers among nearest {best_k} neighbors: {neighbor_result['neighbor_revenue'].sum()}")

display(neighbor_summary.round(4))

# K가 클 수 있으므로 가까운 이웃 표는 상위 20개만 출력한다.
display(neighbor_result.head(20))


target test index: 806
predicted purchase probability: 1.0000
actual revenue: 0
buyers among nearest 151 neighbors: 6


,neighbor_rank,distance,neighbor_revenue
0,1,0.000000,1
1,2,0.027485,1
2,3,0.623814,0
3,4,0.623814,0
4,5,0.686241,1
5,6,0.688914,1
6,7,0.711528,1
7,8,0.802226,0
8,9,0.802226,0
9,10,0.802226,0


**읽는 법**

- 이 예시는 예측 확률이 가장 높은 테스트 샘플을 일부러 고른 것이다. 특이점은 예측 확률이 `1.0`인데 실제 구매 여부는 `0`이라는 점이다.
- 더 특이한 점은 `K=151`인데 구매 이웃은 6명뿐이라는 것이다. 보통 평균만 생각하면 구매 이웃이 적으니 낮은 점수가 나와야 할 것처럼 보인다.
- 하지만 최종 모델은 `distance` 가중치를 쓴다. 가장 가까운 이웃의 거리가 `0`이고 그 이웃이 구매자이면, 멀리 있는 다수의 비구매 이웃보다 이 가까운 구매 이웃이 예측을 강하게 끌어올릴 수 있다.
- 그래서 이 사례는 KNN 확률값의 한계를 잘 보여준다. 모델 점수는 실제 구매 확률이라기보다, 현재 샘플이 구매자와 얼마나 가까운 패턴을 가졌는지 나타내는 거리 기반 우선순위 점수로 해석하는 편이 안전하다.

### 5-5. 최종 해석

**핵심 결론**

최종 KNN Pipeline은 실제 구매 확률을 정밀하게 맞히는 모델이라기보다, 구매 가능성이 높은 세션을 먼저 걸러내는 1차 우선순위 모델로 보는 것이 적절하다.

**근거**

- 예측 점수 상위 decile의 실제 구매율이 최하위 decile보다 훨씬 높다. 즉, 점수가 높을수록 실제 구매 가능성이 높아지는 방향성은 있다.
- 예측 상위 그룹은 상품 페이지를 많이 보고 오래 머물며, `ExitRates`와 `BounceRates`가 낮다. 모델은 사이트 안에서 적극적으로 상품을 탐색한 행동을 구매 신호로 본다.
- permutation 결과에서도 `is_new_visitor`, `ProductRelated_Duration`, `ExitRates`, `ProductRelated`가 중요하게 나왔다. 이는 5-2의 상·하위 그룹 비교와 같은 방향이다.

**주의점**

- 예측 확률 `1.0`이 실제 구매를 보장하지 않는다.
- `distance` 가중치에서는 아주 가까운 구매 이웃 하나가 전체 점수를 크게 끌어올릴 수 있다.
- Brier가 나빠졌기 때문에, 예측값을 보정된 실제 구매 확률로 해석하면 위험하다.

**따라서**

이 모델은 “누가 반드시 구매하는가”를 답하는 모델이 아니라, 마케팅이나 분석 상황에서 “어떤 세션을 먼저 확인할 것인가”를 정하는 데 더 적합하다.